# Half-and-Half modelnet40

This script creates a derivative dataset from ModelNet40 where each point cloud is composed of two halves from both Fullmodelnet40 and partial modelnet40.

We use `pcb-analysis` env for this notebook

The resulting folder `mixedmodelnet40` is available in drive: [`mixedmodelnet40.tar.gz`](https://drive.google.com/drive/folders/1gMLGaXS2_Cm0uIuQvjJEwRt25BzF57dE?usp=drive_link)

In [3]:
import numpy as np

%load_ext autoreload
%autoreload 2

# Set random seed for reproducibility
np.random.seed(42)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
DATA_PATH = "../datasets"

# check if file exists
def dir_exists(path):
    import os
    return os.path.isdir(path)

assert dir_exists(f"{DATA_PATH}/fullmodelnet40"), "Full modelnet40 dataset not found. Please download the dataset and place it in the datasets folder."
assert dir_exists(f"{DATA_PATH}/partialmodelnet40"), "Full modelnet40 dataset not found. Please download the dataset and place it in the datasets folder."

In [57]:
def load_text_as_list(file_path):
    with open(file_path, 'r') as file:
        lines = file.readlines()
    return [line.strip() for line in lines]

def save_text_as_list(file_path, lines):
    with open(file_path, 'w') as file:
        for line in lines:
            file.write(f"{line}\n")

def merge_datasets(full_data, partial_data, prob_full):
    classes = np.unique(full_data["labels"])
    mixed_data = {
        "points": np.empty_like(full_data["points"]),
        "rotations": np.empty_like(full_data["rotations"]),
        "translations": np.empty_like(full_data["translations"]),
        "labels": np.empty_like(full_data["labels"]),
        "filename": []
    }

    for class_id in classes:
        count = np.sum(full_data["labels"] == class_id)
        # generate random indices to sample from partial data of the same class
        size_full = int(count * prob_full)

        class_indices = np.where(full_data["labels"] == class_id)[0]
        full_indices = np.random.choice(class_indices, size=size_full, replace=False)
        partial_indices = np.setdiff1d(class_indices, full_indices)

        mixed_data["points"][full_indices] = full_data["points"][full_indices]
        mixed_data["rotations"][full_indices] = full_data["rotations"][full_indices]
        mixed_data["translations"][full_indices] = full_data["translations"][full_indices]
        mixed_data["points"][partial_indices] = partial_data["points"][partial_indices]
        mixed_data["rotations"][partial_indices] = partial_data["rotations"][partial_indices]
        mixed_data["translations"][partial_indices] = partial_data["translations"][partial_indices]
        mixed_data["labels"][class_indices] = class_id
        filenames = [full_data["filename"][i] + "_full" if i in full_indices else partial_data["filename"][i] + "_partial" for i in class_indices]
        mixed_data["filename"].extend(filenames)

    return mixed_data

In [60]:
# Load train data

# Set probability of selecting full data
PROB_FULL = 0.5 # half full, half partial

full_data = {
    "points": np.load(f"{DATA_PATH}/fullmodelnet40/train_points.npy"),
    "rotations": np.load(f"{DATA_PATH}/fullmodelnet40/train_gt_rot.npy"),
    "translations": np.load(f"{DATA_PATH}/fullmodelnet40/train_gt_tra.npy"),
    "labels": np.load(f"{DATA_PATH}/fullmodelnet40/train_labels.npy"),
    "filename": load_text_as_list(f"{DATA_PATH}/fullmodelnet40/train_filenames.txt")
}
partial_data = {
    "points": np.load(f"{DATA_PATH}/partialmodelnet40/train_points.npy"),
    "rotations": np.load(f"{DATA_PATH}/partialmodelnet40/train_gt_rot.npy"),
    "translations": np.load(f"{DATA_PATH}/partialmodelnet40/train_gt_tra.npy"),
    "labels": np.load(f"{DATA_PATH}/partialmodelnet40/train_labels.npy"),
    "filename": load_text_as_list(f"{DATA_PATH}/partialmodelnet40/partialmodelnet40_train.txt")
}

mixed_data = merge_datasets(full_data, partial_data, PROB_FULL)
full_data, partial_data = None, None  # free memory
# Save mixed dataset
# save mixed_data["filename"] to a text file
save_text_as_list(f"{DATA_PATH}/mixedmodelnet40/mixedmodelnet40_train.txt", mixed_data["filename"])
np.save(f"{DATA_PATH}/mixedmodelnet40/train_points.npy", mixed_data["points"])
np.save(f"{DATA_PATH}/mixedmodelnet40/train_gt_rot.npy", mixed_data["rotations"])
np.save(f"{DATA_PATH}/mixedmodelnet40/train_gt_tra.npy", mixed_data["translations"])
np.save(f"{DATA_PATH}/mixedmodelnet40/train_labels.npy", mixed_data["labels"])

In [ ]:
# Load test data
full_data = {
    "points": np.load(f"{DATA_PATH}/fullmodelnet40/test_points.npy"),
    "rotations": np.load(f"{DATA_PATH}/fullmodelnet40/test_gt_rot.npy"),
    "translations": np.load(f"{DATA_PATH}/fullmodelnet40/test_gt_tra.npy"),
    "labels": np.load(f"{DATA_PATH}/fullmodelnet40/test_labels.npy"),
    "filename": load_text_as_list(f"{DATA_PATH}/fullmodelnet40/test_filenames.txt")
}
partial_data = {
    "points": np.load(f"{DATA_PATH}/partialmodelnet40/test_points.npy"),
    "rotations": np.load(f"{DATA_PATH}/partialmodelnet40/test_gt_rot.npy"),
    "translations": np.load(f"{DATA_PATH}/partialmodelnet40/test_gt_tra.npy"),
    "labels": np.load(f"{DATA_PATH}/partialmodelnet40/test_labels.npy"),
    "filename": load_text_as_list(f"{DATA_PATH}/partialmodelnet40/partialmodelnet40_test.txt")
}

mixed_data = merge_datasets(full_data, partial_data, PROB_FULL)
full_data, partial_data = None, None  # free memory

# Save mixed dataset
save_text_as_list(f"{DATA_PATH}/mixedmodelnet40/mixedmodelnet40_test.txt", mixed_data["filename"])
np.save(f"{DATA_PATH}/mixedmodelnet40/test_points.npy", mixed_data["points"])
np.save(f"{DATA_PATH}/mixedmodelnet40/test_gt_rot.npy", mixed_data["rotations"])
np.save(f"{DATA_PATH}/mixedmodelnet40/test_gt_tra.npy", mixed_data["translations"])
np.save(f"{DATA_PATH}/mixedmodelnet40/test_labels.npy", mixed_data["labels"])

print(f"New dataset successfully created at {DATA_PATH}/mixedmodelnet40")